# 03 — Construcción manual guiada del RDF v0 desde CSV

**Objetivo del notebook:** transformar progresivamente `kg_v0_test_samples.csv` en un primer RDF/Turtle de prueba para EMPKG-lite v0.

Este notebook es una fase de aprendizaje y validación antes de consolidar la lógica en un script reproducible.

## Qué debe generar este notebook

Entrada:

```text
data/samples/kg_v0_test_samples.csv
```

Salida esperada:

```text
data/processed/empkg_v0_test.ttl
```

## Alcance de v0

Incluye:

- `Sample`
- `Study`
- `Location`
- `EMPOCategory`
- `EnvironmentDescription`

No incluye todavía:

- ASVs
- Taxones
- Abundancias muestra-taxón
- Mapeo ENVO real
- Mapeo GAZ real
- Mapeo NCBITaxon
- Procesamiento LLM
- GraphDB
- SHACL

## 1. Imports y rutas

En esta sección se preparan las librerías y rutas del proyecto.

**Idea:** mantener el notebook ejecutable desde `notebooks/`, pero usando rutas relativas a la raíz del repositorio.

In [45]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
import re

import pandas as pd
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, XSD

In [46]:
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

INPUT_CSV_PATH = PROJECT_ROOT / "data" / "samples" / "kg_v0_test_samples.csv"
OUTPUT_TTL_PATH = PROJECT_ROOT / "data" / "processed" / "empkg_v0_test.ttl"

print(f"Project root: {PROJECT_ROOT}")
print(f"Entrada:      {INPUT_CSV_PATH}")
print(f"Salida:       {OUTPUT_TTL_PATH}")

if not INPUT_CSV_PATH.exists():
    raise FileNotFoundError(f"No existe el CSV de entrada: {INPUT_CSV_PATH}")

Project root: /home/oier/EMPKG
Entrada:      /home/oier/EMPKG/data/samples/kg_v0_test_samples.csv
Salida:       /home/oier/EMPKG/data/processed/empkg_v0_test.ttl


## 2. Carga e inspección inicial del CSV

Antes de generar RDF, comprueba que el CSV contiene exactamente lo que se espera.

- ¿Hay 17 filas?
- ¿Existe `sample_id` como columna?
- ¿Hay `sample_id` duplicados?
- ¿Hay 17 categorías distintas en `empo_3`?
- ¿Qué columnas contienen valores ausentes?

In [47]:
df = pd.read_csv(INPUT_CSV_PATH)
print(f"Shape: {df.shape}")
display(df.head())

Shape: (17, 24)


,sample_id,study_id,empo_1,empo_2,empo_3,env_biome,env_feature,env_material,envo_biome_0,envo_biome_1,...,longitude_deg,depth_m,altitude_m,elevation_m,temperature_deg_c,ph,salinity_psu,oxygen_mg_per_l,biom_total_reads,biom_observed_asvs
0,1001.SKB7,1001,Host-associated,Plant,Plant rhizosphere,cropland biome,plant-associated habitat,organic material,biome,terrestrial biome,...,-117.241000,0.15,0.0,114.00,15.0000,6.940,7.1500,NaN,5000,207
1,1883.2011.491.Crump.Artic.LTREB.main.lane4.NoI...,1883,Free-living,Saline,Water (saline),marine biome,coastal water body,coastal water,biome,aquatic biome,...,-143.190383,0.50,0.0,0.00,5.2000,7.800,30.0000,13.2,5000,94
2,1041.M041.100,1041,Free-living,Non-saline,Water (non-saline),Large lake biome,freshwater lake,fresh water,biome,aquatic biome,...,-86.721000,100.00,0.0,98.61,3.5477,8.203,0.1359,NaN,5000,336
3,1001.SKD6,1001,Free-living,Non-saline,Soil (non-saline),cropland biome,plant-associated habitat,soil,biome,terrestrial biome,...,-117.241000,0.15,0.0,114.00,15.0000,6.800,7.1000,NaN,5000,1045
4,807.C.F.10.b,807,Free-living,Non-saline,Sediment (non-saline),Small river biome,fresh water,stream sediment,biome,aquatic biome,...,-106.949183,0.00,0.0,1058.71,10.5000,8.690,NaN,NaN,5000,1328


In [48]:
EXPECTED_COLUMNS = [
    "sample_id",
    "study_id",
    "empo_1",
    "empo_2",
    "empo_3",
    "env_biome",
    "env_feature",
    "env_material",
    "envo_biome_0",
    "envo_biome_1",
    "envo_biome_2",
    "envo_biome_3",
    "country",
    "latitude_deg",
    "longitude_deg",
    "depth_m",
    "altitude_m",
    "elevation_m",
    "temperature_deg_c",
    "ph",
    "salinity_psu",
    "oxygen_mg_per_l",
    "biom_total_reads",
    "biom_observed_asvs",
]

missing_columns = [col for col in EXPECTED_COLUMNS if col not in df.columns]
extra_columns = [col for col in df.columns if col not in EXPECTED_COLUMNS]

print(f"Columnas esperadas: {len(EXPECTED_COLUMNS)}")
print(f"Columnas reales:    {len(df.columns)}")

if missing_columns:
    raise ValueError(f"Faltan columnas esperadas: {missing_columns}")

if extra_columns:
    print(f"AVISO: hay columnas extra no esperadas: {extra_columns}")

if df.shape[0] != 17:
    raise ValueError(f"Se esperaban 17 muestras, pero hay {df.shape[0]}")

if df["sample_id"].duplicated().any():
    duplicated = df.loc[df["sample_id"].duplicated(), "sample_id"].tolist()
    raise ValueError(f"Hay sample_id duplicados: {duplicated}")

print("OK: estructura básica del CSV validada.")

Columnas esperadas: 24
Columnas reales:    24
OK: estructura básica del CSV validada.


In [49]:
# Comprobaciones exploratorias:
print("Número de muestras:", df["sample_id"].nunique())
print("Número de estudios:", df["study_id"].nunique())
print("Número de categorías empo_3:", df["empo_3"].nunique())

print("\nCategorías empo_3:")
display(df["empo_3"].value_counts().sort_index())

missing_summary = df.isna().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)

print("Columnas con valores ausentes:")
display(missing_summary)

Número de muestras: 17
Número de estudios: 15
Número de categorías empo_3: 17

Categorías empo_3:


empo_3
Aerosol (non-saline)     1
Animal corpus            1
Animal distal gut        1
Animal proximal gut      1
Animal secretion         1
Animal surface           1
Hypersaline (saline)     1
Plant corpus             1
Plant rhizosphere        1
Plant surface            1
Sediment (non-saline)    1
Sediment (saline)        1
Soil (non-saline)        1
Surface (non-saline)     1
Surface (saline)         1
Water (non-saline)       1
Water (saline)           1
Name: count, dtype: int64

Columnas con valores ausentes:


oxygen_mg_per_l      16
salinity_psu         12
ph                    7
temperature_deg_c     6
envo_biome_3          3
altitude_m            1
dtype: int64

## 3. Diseño RDF v0 que vamos a implementar

Modelo conceptual:

```text
Sample ──belongsToStudy──────────────> Study
Sample ──wasCollectedAt──────────────> Location
Sample ──hasEMPOCategory─────────────> EMPOCategory
Sample ──hasEnvironmentDescription───> EnvironmentDescription
```

Regla principal:

```text
Si un valor está ausente, no se genera triple.
```

## Decisiones de modelado

| Elemento CSV | Representación RDF |
|---|---|
| `sample_id` | URI de `Sample` |
| `study_id` | URI de `Study` reutilizable |
| `country`, `latitude_deg`, `longitude_deg`, `depth_m`, `altitude_m`, `elevation_m` | URI de `Location` reutilizable |
| `empo_3` | URI de `EMPOCategory` reutilizable |
| `env_*`, `envo_biome_*` | literales de `EnvironmentDescription` |
| medidas físico-químicas | literales de `Sample` |
| `biom_total_reads`, `biom_observed_asvs` | literales enteros de `Sample` |

## 4. Namespaces y prefijos

En esta sección se define el vocabulario propio y los namespaces de recursos.

Se recomienda separar:

- `empkg:` para clases y propiedades.
- `sample:`, `study:`, `loc:`, `empo:` y `envdesc:` para recursos concretos.

In [50]:
EMPKG = Namespace("https://w3id.org/empkg/ontology/")

SAMPLE = Namespace("https://w3id.org/empkg/resource/sample/")
STUDY = Namespace("https://w3id.org/empkg/resource/study/")
LOC = Namespace("https://w3id.org/empkg/resource/location/")
EMPO = Namespace("https://w3id.org/empkg/resource/empo-category/")
ENVDESC = Namespace("https://w3id.org/empkg/resource/environment-description/")

SCHEMA = Namespace("https://schema.org/")

g = Graph()

g.bind("empkg", EMPKG)
g.bind("sample", SAMPLE)
g.bind("study", STUDY)
g.bind("loc", LOC)
g.bind("empo", EMPO)
g.bind("envdesc", ENVDESC)

g.bind("rdf", RDF)
g.bind("rdfs", RDFS)
g.bind("xsd", XSD)
g.bind("schema", SCHEMA)

print(f"Triples en el grafo: {len(g)}")

print("Prefijos principales registrados:")
for prefix, namespace in g.namespaces():
    if prefix in {"empkg", "sample", "study", "loc", "empo", "envdesc", "rdf", "rdfs", "xsd", "schema"}:
        print(f"{prefix}: {namespace}")

Triples en el grafo: 0
Prefijos principales registrados:
schema: https://schema.org/
rdf: http://www.w3.org/1999/02/22-rdf-syntax-ns#
rdfs: http://www.w3.org/2000/01/rdf-schema#
xsd: http://www.w3.org/2001/XMLSchema#
empkg: https://w3id.org/empkg/ontology/
sample: https://w3id.org/empkg/resource/sample/
study: https://w3id.org/empkg/resource/study/
loc: https://w3id.org/empkg/resource/location/
empo: https://w3id.org/empkg/resource/empo-category/
envdesc: https://w3id.org/empkg/resource/environment-description/


## 5. Capa mínima de modelo RDF

Esta capa no es obligatoria para que Turtle sea válido, pero ayuda a documentar el KG.

Debe declarar:

Clases:

- `empkg:Sample`
- `empkg:Study`
- `empkg:Location`
- `empkg:EMPOCategory`
- `empkg:EnvironmentDescription`

Propiedades principales:

- `empkg:belongsToStudy`
- `empkg:wasCollectedAt`
- `empkg:hasEMPOCategory`
- `empkg:hasEnvironmentDescription`

In [51]:
CLASSES = {
    "Sample": "Sample",
    "Study": "Study",
    "Location": "Location",
    "EMPOCategory": "EMPO category",
    "EnvironmentDescription": "Environment description",
}
for class_name, label in CLASSES.items():
    class_uri = EMPKG[class_name]
    g.add((class_uri, RDF.type, RDFS.Class))
    g.add((class_uri, RDFS.label, Literal(label)))


RELATIONS = {
    "belongsToStudy": {
        "domain": "Sample",
        "range": "Study",
        "label": "belongs to study",
    },
    "wasCollectedAt": {
        "domain": "Sample",
        "range": "Location",
        "label": "was collected at",
    },
    "hasEMPOCategory": {
        "domain": "Sample",
        "range": "EMPOCategory",
        "label": "has EMPO category",
    },
    "hasEnvironmentDescription": {
        "domain": "Sample",
        "range": "EnvironmentDescription",
        "label": "has environment description",
    },
}

for property_name, info in RELATIONS.items():
    property_uri = EMPKG[property_name]
    domain_uri = EMPKG[info["domain"]]
    range_uri = EMPKG[info["range"]]

    g.add((property_uri, RDF.type, RDF.Property))
    g.add((property_uri, RDFS.domain, domain_uri))
    g.add((property_uri, RDFS.range, range_uri))
    g.add((property_uri, RDFS.label, Literal(info["label"])))

print(f"Triples tras añadir la capa de modelo: {len(g)}")
print(g.serialize(format="turtle"))

Triples tras añadir la capa de modelo: 26
@prefix empkg: <https://w3id.org/empkg/ontology/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

empkg:EMPOCategory a rdfs:Class ;
    rdfs:label "EMPO category" .

empkg:EnvironmentDescription a rdfs:Class ;
    rdfs:label "Environment description" .

empkg:Location a rdfs:Class ;
    rdfs:label "Location" .

empkg:Sample a rdfs:Class ;
    rdfs:label "Sample" .

empkg:Study a rdfs:Class ;
    rdfs:label "Study" .

empkg:belongsToStudy a rdf:Property ;
    rdfs:label "belongs to study" ;
    rdfs:domain empkg:Sample ;
    rdfs:range empkg:Study .

empkg:hasEMPOCategory a rdf:Property ;
    rdfs:label "has EMPO category" ;
    rdfs:domain empkg:Sample ;
    rdfs:range empkg:EMPOCategory .

empkg:hasEnvironmentDescription a rdf:Property ;
    rdfs:label "has environment description" ;
    rdfs:domain empkg:Sample ;
    rdfs:range empkg:EnvironmentDescription .

empkg:wasCol

## 6. Funciones de limpieza y sanitización

Aquí se definen funciones pequeñas y fáciles de probar.

Objetivo:

- Detectar valores ausentes.
- Convertir IDs y textos en fragmentos seguros de URI.
- Convertir números de coordenadas en texto seguro para URIs de `Location`.

Ejemplos esperados:

```text
1883.2011.491.Crump.Artic.LTREB.main.lane4.NoIndex
→ 1883_2011_491_Crump_Artic_LTREB_main_lane4_NoIndex

Water (saline)
→ water_saline

Soil (non-saline)
→ soil_non_saline

-143.1903833
→ m143_1903833
```

In [52]:
MISSING_STRINGS = {
    "",
    "nan",
    "none",
    "null",
    "na",
    "n/a",
    "unknown",
    "missing: not provided",
    "not applicable",
    "not provided",
}

def is_missing(value) -> bool:
    """
    Devuelve True si el valor debe considerarse ausente.

    Se usa antes de generar triples:
    si un valor está ausente, no se añade al grafo.
    """
    if pd.isna(value):
        return True

    if isinstance(value, str):
        clean = value.strip().lower()
        return clean in MISSING_STRINGS
    
    return False

In [53]:
missing_tests = [
    None,
    pd.NA,
    float("nan"),
    "",
    " ",
    "unknown",
    "Unknown",
    "Missing: Not provided",
    "marine biome",
]

for value in missing_tests:
    print(f"{value!r} -> {is_missing(value)}")

None -> True
<NA> -> True
nan -> True
'' -> True
' ' -> True
'unknown' -> True
'Unknown' -> True
'Missing: Not provided' -> True
'marine biome' -> False


In [54]:
def sanitize_id_for_uri(value: str) -> str:
    """
    Sanitiza identificadores técnicos conservando mayúsculas/minúsculas.

    Útil para:
        sample_id
        study_id
        environment-description basado en sample_id
    """
    if is_missing(value):
        raise ValueError("No se puede sanitizar un valor ausente para URI.")
    
    text = str(value).strip()
    
    # Sustituir cualquier carácter no alfanumérico por "_".
    text = re.sub(r"[^a-zA-Z0-9]+", "_", text)

    # Eliminar "_" sobrantes y varios "_" seguidos
    text = text.strip("_")
    text = re.sub(r"_+", "_", text)

    return text

def sanitize_label_for_uri(value: str) -> str:
    """
    Sanitiza etiquetas textuales normalizadas a minúsculas.

    Útil para:
        empo_3
        otros valores categóricos
    """
    if is_missing(value):
        raise ValueError("No se puede sanitizar un valor ausente para URI.")
    
    text = str(value).strip().lower()
    
    text = text.replace("(non-saline)", "non saline")
    text = text.replace("(saline)", "saline")

    text = re.sub(r"[^a-zA-Z0-9]+", "_", text)
    text = text.strip("_")
    text = re.sub(r"_+", "_", text)

    return text
    
example_row = df.iloc[1]
print(example_row['sample_id'], sanitize_id_for_uri(example_row['sample_id']), sep="\n")
print(example_row['empo_3'], sanitize_label_for_uri(example_row['empo_3'],), sep="\n")


1883.2011.491.Crump.Artic.LTREB.main.lane4.NoIndex
1883_2011_491_Crump_Artic_LTREB_main_lane4_NoIndex
Water (saline)
water_saline


In [55]:
def sanitize_number_for_uri(value) -> str:
    """
    Convierte un número en un fragmento seguro para URI.

    Ejemplos:
        70.06601667  -> "70_06601667"
        -143.1903833 -> "m143_1903833"
        0.5          -> "0_5"

    Si el valor está ausente, devuelve "na" para mantener una URI determinista.
    """
    if is_missing(value):
        return "na"

    try:
        number = Decimal(str(value).strip())
    except (InvalidOperation, ValueError):
        raise ValueError(f"No se puede sanitizar como número para URI: {value!r}")

    text = format(number, "f")

    if text.startswith("-"):
        text = "m" + text[1:]

    text = text.replace(".", "_")

    return text

print(example_row["latitude_deg"], sanitize_number_for_uri(example_row["latitude_deg"]), sep="\n")
print(example_row["longitude_deg"], sanitize_number_for_uri(example_row["longitude_deg"]), sep="\n")
print(example_row["depth_m"], sanitize_number_for_uri(example_row["depth_m"]), sep="\n")
print(example_row["altitude_m"], sanitize_number_for_uri(example_row["altitude_m"]), sep="\n")
print(example_row["elevation_m"], sanitize_number_for_uri(example_row["elevation_m"]), sep="\n")

70.06601667
70_06601667
-143.1903833
m143_1903833
0.5
0_5
0.0
0_0
0.0
0_0


In [56]:
test_values = [
    "Water (saline)",
    "Water (non-saline)",
    "Soil (non-saline)",
    "Animal distal gut",
    "1883.2011.491.Crump.Artic.LTREB.main.lane4.NoIndex",
    "1001.SKB7",
]

for value in test_values:
    print(f"{value!r}")
    print("  id uri:    ", sanitize_id_for_uri(value))
    print("  label uri: ", sanitize_label_for_uri(value))

'Water (saline)'
  id uri:     Water_saline
  label uri:  water_saline
'Water (non-saline)'
  id uri:     Water_non_saline
  label uri:  water_non_saline
'Soil (non-saline)'
  id uri:     Soil_non_saline
  label uri:  soil_non_saline
'Animal distal gut'
  id uri:     Animal_distal_gut
  label uri:  animal_distal_gut
'1883.2011.491.Crump.Artic.LTREB.main.lane4.NoIndex'
  id uri:     1883_2011_491_Crump_Artic_LTREB_main_lane4_NoIndex
  label uri:  1883_2011_491_crump_artic_ltreb_main_lane4_noindex
'1001.SKB7'
  id uri:     1001_SKB7
  label uri:  1001_skb7


In [57]:
def sanitize_country_for_location_uri(value) -> str:
    """
    Sanitiza el país para usarlo dentro de una URI de Location.

    En v0, country se guarda como literal original.
    Esta función solo genera una parte corta para la URI.
    """
    if is_missing(value):
        return "country_na"

    text = str(value).strip()

    # Quitar prefijo GAZ: si aparece.
    if text.startswith("GAZ:"):
        text = text.replace("GAZ:", "", 1)

    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9]+", "_", text)
    text = text.strip("_")
    text = re.sub(r"_+", "_", text)

    return text

In [58]:
number_tests = [
    70.06601667,
    -143.1903833,
    0.5,
    0.0,
    pd.NA,
    None,
    "",
]

for value in number_tests:
    print(f"{value!r} -> {sanitize_number_for_uri(value)}")

70.06601667 -> 70_06601667
-143.1903833 -> m143_1903833
0.5 -> 0_5
0.0 -> 0_0
<NA> -> na
None -> na
'' -> na


## 7. Constructores de URIs

Estas funciones encapsulan cómo se transforma cada elemento del CSV en una URI RDF.



- `make_sample_uri(sample_id)`
- `make_study_uri(study_id)`
- `make_empo_uri(empo_3)`
- `make_environment_description_uri(sample_id)`
- `make_location_uri(row)`
- `make_location_uri(label)`


In [59]:
def make_sample_uri(sample_id: str) -> URIRef:
    """
    URI para el nodo Sample.

    Ejemplo: "1001.SKB7" → sample:1001_SKB7
    Usa sanitize_id_for_uri porque preserva mayúsculas/minúsculas del ID original.
    """
    return SAMPLE[sanitize_id_for_uri(sample_id)]

In [60]:
def make_study_uri(study_id) -> URIRef:
    """
    URI para el nodo Study.

    Varias muestras con el mismo study_id generan la misma URI.
    No asumimos que study_id sea necesariamente entero en todos los datasets.
    """
    return STUDY[sanitize_id_for_uri(study_id)]

In [61]:
def make_empo_uri(empo_3: str) -> URIRef:
    """
    URI para el nodo EMPOCategory.

    Ejemplo: "Water (saline)"     → empo:water_saline
             "Water (non-saline)" → empo:water_non_saline
             "Soil (non-saline)"  → empo:soil_non_saline

    Usa sanitize_label_for_uri porque empo_3 es una etiqueta controlada.
    """
    return EMPO[sanitize_label_for_uri(empo_3)]

In [62]:
def make_environment_description_uri(sample_id: str) -> URIRef:
    """
    URI para el nodo EnvironmentDescription.

    En v0 es 1:1 con Sample, así que comparte su ID sanitizado,
    pero dentro del namespace envdesc:.
    """
    return ENVDESC[sanitize_id_for_uri(sample_id)]

In [63]:
def make_location_uri(row) -> URIRef:
    """
    URI para el nodo Location, construida de forma determinista
    a partir de todos los campos geográficos.
    
    Si dos muestras tienen exactamente los mismos valores geográficos,
    obtienen la misma URI → el mismo nodo en el grafo → deduplicación automática.
    
    Si un campo está ausente, se incluye la cadena "na" en su posición.
    Esto diferencia una ubicación con depth=0.0 de otra con depth ausente,
    lo cual es semánticamente correcto.
    
    Formato:
        loc:{country}_lat{lat}_lon{lon}_depth{depth}_alt{alt}_elev{elev}
    
    Ejemplo:
        loc:united_states_of_america_lat33_194_lonm117_241_depth0_15_alt0_0_elev114_0
    """
    country = sanitize_country_for_location_uri(row['country'])
    lat = f"lat{sanitize_number_for_uri(row['latitude_deg'])}"
    lon = f"lon{sanitize_number_for_uri(row['longitude_deg'])}"
    depth = f"depth{sanitize_number_for_uri(row['depth_m'])}"
    alt = f"alt{sanitize_number_for_uri(row['altitude_m'])}"
    elev = f"elev{sanitize_number_for_uri(row['elevation_m'])}"

    key = "_".join([country, lat, lon, depth, alt, elev])
    return LOC[key]


In [64]:
def make_location_label(row) -> str:
    """
    Construye una etiqueta legible para Location.
    """
    lat = row["latitude_deg"]
    lon = row["longitude_deg"]

    if not is_missing(lat) and not is_missing(lon):
        return f"Location {lat}, {lon}"

    country = row["country"]
    if not is_missing(country):
        return f"Location {country}"

    return "Location"

In [65]:
for i in [0, 1, 2, 3]:
    row = df.iloc[i]
    print(f"[fila {i}] sample_id: {row['sample_id']}")
    print(f"  make_sample_uri:      {make_sample_uri(row['sample_id'])}")
    print(f"  make_study_uri:       {make_study_uri(row['study_id'])}")
    print(f"  make_empo_uri:        {make_empo_uri(row['empo_3'])}")
    print(f"  make_envdesc_uri:     {make_environment_description_uri(row['sample_id'])}")
    print(f"  make_location_uri:    {make_location_uri(row)}")
    print()

[fila 0] sample_id: 1001.SKB7
  make_sample_uri:      https://w3id.org/empkg/resource/sample/1001_SKB7
  make_study_uri:       https://w3id.org/empkg/resource/study/1001
  make_empo_uri:        https://w3id.org/empkg/resource/empo-category/plant_rhizosphere
  make_envdesc_uri:     https://w3id.org/empkg/resource/environment-description/1001_SKB7
  make_location_uri:    https://w3id.org/empkg/resource/location/united_states_of_america_lat33_194_lonm117_241_depth0_15_alt0_0_elev114_0

[fila 1] sample_id: 1883.2011.491.Crump.Artic.LTREB.main.lane4.NoIndex
  make_sample_uri:      https://w3id.org/empkg/resource/sample/1883_2011_491_Crump_Artic_LTREB_main_lane4_NoIndex
  make_study_uri:       https://w3id.org/empkg/resource/study/1883
  make_empo_uri:        https://w3id.org/empkg/resource/empo-category/water_saline
  make_envdesc_uri:     https://w3id.org/empkg/resource/environment-description/1883_2011_491_Crump_Artic_LTREB_main_lane4_NoIndex
  make_location_uri:    https://w3id.org/empkg

## 8. Funciones auxiliares para literales

La idea es no repetir la misma lógica muchas veces.

Funciones recomendadas:

- `add_text_literal_if_present(graph, subject, predicate, value)`
- `add_decimal_literal_if_present(graph, subject, predicate, value)`
- `add_integer_literal_if_present(graph, subject, predicate, value)`

Regla:

```text
Si el valor está ausente, no se añade ningún triple.
```

In [66]:

def add_text_literal_if_present(graph: Graph, subject: URIRef, predicate: URIRef, value) -> None:
    """
    Añade un literal textual solo si el valor existe.

    No convierte NaN en "nan".
    No añade cadenas vacías.
    """
    if is_missing(value):
        return
    
    text = str(value).strip()

    if text == "":
        return
    
    graph.add((subject, predicate, Literal(text)))



In [67]:

def add_decimal_literal_if_present(graph: Graph, subject: URIRef, predicate: URIRef, value) -> None:
    """
    Añade un literal xsd:decimal solo si el valor existe y es numérico.
    """
    if is_missing(value):
        return
    
    try:
        decimal_value = Decimal(str(value).strip())
    except (InvalidOperation, ValueError):
        raise ValueError(
            f"No se puede convertir a xsd:decimal: {value!r} "
            f"para el predicado {predicate}"
        )
    
    graph.add((subject, predicate, Literal(decimal_value, datatype=XSD.decimal)))

In [68]:
def add_integer_literal_if_present(graph: Graph, subject: URIRef, predicate: URIRef, value) -> None:
    """
    Añade un literal xsd:integer solo si el valor existe y es entero.
    """
    if is_missing(value):
        return

    try:
        decimal_value = Decimal(str(value).strip())
    except (InvalidOperation, ValueError):
        raise ValueError(
            f"No se puede convertir a xsd:integer: {value!r} "
            f"para el predicado {predicate}"
        )

    if decimal_value != decimal_value.to_integral_value():
        raise ValueError(
            f"El valor no es entero: {value!r} "
            f"para el predicado {predicate}"
        )

    graph.add((subject, predicate, Literal(int(decimal_value), datatype=XSD.integer)))

## 9. Añadir `Study`

`Study` es un nodo reutilizable.

Si varias muestras comparten `study_id`, deben apuntar al mismo nodo `Study`.

In [69]:
def add_study(graph: Graph, row) -> URIRef:
    """
    Añade los triples del nodo Study y devuelve su URI.

    RDFLib no duplica triples idénticos, así que no hace falta comprobar
    si ya existe.
    """
    study_id = row["study_id"]
    study_uri = make_study_uri(study_id)

    graph.add((study_uri, RDF.type, EMPKG.Study))
    add_text_literal_if_present(graph, study_uri, RDFS.label, f"Study {study_id}")
    add_text_literal_if_present(graph, study_uri, EMPKG.studyId, study_id)

    return study_uri

## 10. Añadir `EMPOCategory`

`EMPOCategory` es un nodo reutilizable basado en `empo_3`.

Debe conservar el texto original en:

- `rdfs:label`
- `empkg:empo3`

Pero la URI debe estar sanitizada.

In [70]:
def add_empo_category(graph: Graph, row) -> URIRef:
    """
    Añade los triples de EMPOCategory y devuelve su URI.

    La URI se basa en empo_3 sanitizado.
    El texto original se conserva en rdfs:label y empkg:empo3.
    """
    empo_uri = make_empo_uri(row["empo_3"])

    graph.add((empo_uri, RDF.type, EMPKG.EMPOCategory))
    add_text_literal_if_present(graph, empo_uri, RDFS.label, row["empo_3"])
    add_text_literal_if_present(graph, empo_uri, EMPKG.empo1, row["empo_1"])
    add_text_literal_if_present(graph, empo_uri, EMPKG.empo2, row["empo_2"])
    add_text_literal_if_present(graph, empo_uri, EMPKG.empo3, row["empo_3"])

    return empo_uri

## 11. Añadir `Location`

`Location` es un nodo reutilizable basado en los campos geográficos.

Debe incluir:

- `rdf:type empkg:Location`
- `rdfs:label`
- `empkg:country`
- `schema:latitude`
- `schema:longitude`
- `empkg:depthM`
- `empkg:altitudeM`
- `empkg:elevationM`

Solo se añaden los literales si existen.

In [71]:

def add_location(graph: Graph, row) -> URIRef:
    """
    Añade los triples de Location y devuelve su URI.

    Location es reutilizable: dos filas con la misma clave geográfica
    producen la misma URI.
    """
    loc_uri = make_location_uri(row)

    graph.add((loc_uri, RDF.type, EMPKG.Location))

    add_text_literal_if_present(graph, loc_uri, RDFS.label, make_location_label(row))
    add_text_literal_if_present(graph, loc_uri, EMPKG.country, row["country"])

    add_decimal_literal_if_present(graph, loc_uri, SCHEMA.latitude, row["latitude_deg"])
    add_decimal_literal_if_present(graph, loc_uri, SCHEMA.longitude, row["longitude_deg"])
    add_decimal_literal_if_present(graph, loc_uri, EMPKG.depthM, row["depth_m"])
    add_decimal_literal_if_present(graph, loc_uri, EMPKG.altitudeM, row["altitude_m"])
    add_decimal_literal_if_present(graph, loc_uri, EMPKG.elevationM, row["elevation_m"])
    return loc_uri

## 12. Añadir `EnvironmentDescription`

En v0 este nodo es 1:1 con `Sample`.

No se deduplica todavía.

Debe conservar los textos originales:

- `env_biome`
- `env_feature`
- `env_material`
- `envo_biome_0`
- `envo_biome_1`
- `envo_biome_2`
- `envo_biome_3`

In [72]:
def add_environment_description(graph: Graph, row) -> URIRef:
    """
    Añade los triples de EnvironmentDescription y devuelve su URI.

    En v0 es 1:1 con Sample.
    Los campos se guardan como literales originales, sin mapear a ENVO real.
    """
    env_uri = make_environment_description_uri(row["sample_id"])

    graph.add((env_uri, RDF.type, EMPKG.EnvironmentDescription))

    add_text_literal_if_present(graph, env_uri, EMPKG.envBiome, row["env_biome"])
    add_text_literal_if_present(graph, env_uri, EMPKG.envFeature, row["env_feature"])
    add_text_literal_if_present(graph, env_uri, EMPKG.envMaterial, row["env_material"])

    add_text_literal_if_present(graph, env_uri, EMPKG.envoBiome0, row["envo_biome_0"])
    add_text_literal_if_present(graph, env_uri, EMPKG.envoBiome1, row["envo_biome_1"])
    add_text_literal_if_present(graph, env_uri, EMPKG.envoBiome2, row["envo_biome_2"])
    add_text_literal_if_present(graph, env_uri, EMPKG.envoBiome3, row["envo_biome_3"])

    return env_uri

## 13. Añadir `Sample`

`Sample` es el nodo central.

Debe enlazar con:

- `Study`
- `Location`
- `EMPOCategory`
- `EnvironmentDescription`

Y debe guardar literales propios:

- `sampleId`
- `temperatureDegC`
- `ph`
- `salinityPsu`
- `oxygenMgPerL`
- `biomTotalReads`
- `biomObservedASVs`

In [73]:
SAMPLE_DECIMAL_PROPERTIES = {
    "temperature_deg_c": EMPKG.temperatureDegC,
    "ph": EMPKG.ph,
    "salinity_psu": EMPKG.salinityPsu,
    "oxygen_mg_per_l": EMPKG.oxygenMgPerL,
}

SAMPLE_INTEGER_PROPERTIES = {
    "biom_total_reads": EMPKG.biomTotalReads,
    "biom_observed_asvs": EMPKG.biomObservedASVs,
}

def add_sample(graph: Graph, row) -> URIRef:
    """
    Añade el nodo Sample, sus relaciones principales y sus literales propios.
    """
    sample_id = row['sample_id']
    sample_uri = make_sample_uri(sample_id)

    study_uri = add_study(graph, row)
    empo_uri = add_empo_category(graph, row)
    loc_uri = add_location(graph, row)
    env_uri = add_environment_description(graph, row)

    graph.add((sample_uri, RDF.type, EMPKG.Sample))

    add_text_literal_if_present(graph, sample_uri, RDFS.label, f"Sample {sample_id}")
    add_text_literal_if_present(graph, sample_uri, EMPKG['sampleId'], sample_id)

    graph.add((sample_uri, EMPKG.belongsToStudy, study_uri))
    graph.add((sample_uri, EMPKG.hasEMPOCategory, empo_uri))
    graph.add((sample_uri, EMPKG.wasCollectedAt, loc_uri))
    graph.add((sample_uri, EMPKG.hasEnvironmentDescription, env_uri))

    for csv_column, rdf_property in SAMPLE_DECIMAL_PROPERTIES.items():
        add_decimal_literal_if_present(graph, sample_uri, rdf_property, row[csv_column])

    for csv_column, rdf_property in SAMPLE_INTEGER_PROPERTIES.items():
        add_integer_literal_if_present(graph, sample_uri, rdf_property, row[csv_column])

    return sample_uri

    
    

## 14. Prueba con una sola muestra

Antes de procesar las 17 muestras, genera RDF para una sola fila y compáralo conceptualmente con tu Turtle manual.

Preguntas:

- ¿Se parece al `.ttl` que escribiste a mano?
- ¿Tiene las 4 relaciones principales?
- ¿Omite valores ausentes?
- ¿Las URIs son legibles y estables?

In [74]:
def create_empty_graph() -> Graph:
    """
    Crea un grafo vacío con todos los prefijos registrados.
    """
    graph = Graph()

    graph.bind("empkg", EMPKG)
    graph.bind("sample", SAMPLE)
    graph.bind("study", STUDY)
    graph.bind("loc", LOC)
    graph.bind("empo", EMPO)
    graph.bind("envdesc", ENVDESC)

    graph.bind("rdf", RDF)
    graph.bind("rdfs", RDFS)
    graph.bind("xsd", XSD)
    graph.bind("schema", SCHEMA)

    return graph


def add_model_layer(graph: Graph) -> None:
    """
    Añade clases y propiedades principales del modelo v0.
    """
    for class_name, label in CLASSES.items():
        class_uri = EMPKG[class_name]
        graph.add((class_uri, RDF.type, RDFS.Class))
        graph.add((class_uri, RDFS.label, Literal(label)))

    for property_name, info in RELATIONS.items():
        property_uri = EMPKG[property_name]
        domain_uri = EMPKG[info["domain"]]
        range_uri = EMPKG[info["range"]]

        graph.add((property_uri, RDF.type, RDF.Property))
        graph.add((property_uri, RDFS.domain, domain_uri))
        graph.add((property_uri, RDFS.range, range_uri))
        graph.add((property_uri, RDFS.label, Literal(info["label"])))


g_test = create_empty_graph()
add_model_layer(g_test)

sample_uri = add_sample(g_test, df.iloc[1])

print(g_test.serialize(format="turtle"))

@prefix empkg: <https://w3id.org/empkg/ontology/> .
@prefix empo: <https://w3id.org/empkg/resource/empo-category/> .
@prefix envdesc: <https://w3id.org/empkg/resource/environment-description/> .
@prefix loc: <https://w3id.org/empkg/resource/location/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sample: <https://w3id.org/empkg/resource/sample/> .
@prefix schema: <https://schema.org/> .
@prefix study: <https://w3id.org/empkg/resource/study/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

empkg:EMPOCategory a rdfs:Class ;
    rdfs:label "EMPO category" .

empkg:EnvironmentDescription a rdfs:Class ;
    rdfs:label "Environment description" .

empkg:Location a rdfs:Class ;
    rdfs:label "Location" .

empkg:Sample a rdfs:Class ;
    rdfs:label "Sample" .

empkg:Study a rdfs:Class ;
    rdfs:label "Study" .

empkg:belongsToStudy a rdf:Property ;
    rdfs:label "belongs to study" ;
    rdfs:domain empkg:

In [75]:
def find_bad_literals(graph: Graph):
    """
    Busca literales que no deberían aparecer en el RDF.
    """
    bad_values = {"", "nan", "none", "null", "unknown"}

    bad_literals = []

    for s, p, o in graph:
        if isinstance(o, Literal):
            lexical = str(o).strip().lower()
            if lexical in bad_values:
                bad_literals.append((s, p, o))

    return bad_literals

bad_literals = find_bad_literals(g_test)

print(f"Literales problemáticos encontrados: {len(bad_literals)}")

for triple in bad_literals:
    print(triple)

Literales problemáticos encontrados: 0


In [76]:
def print_literal_datatypes(graph: Graph, predicates: list[URIRef]) -> None:
    """
    Muestra los valores y datatypes reales de ciertos predicados.
    Útil porque Turtle puede ocultar xsd:integer/xsd:decimal usando sintaxis abreviada.
    """
    for predicate in predicates:
        print(f"\n{predicate}")
        for _, _, literal in graph.triples((None, predicate, None)):
            if isinstance(literal, Literal):
                print(f"  {literal!r} -> {literal.datatype}")

print_literal_datatypes(
    g_test,
    [
        EMPKG.temperatureDegC,
        EMPKG.ph,
        EMPKG.salinityPsu,
        EMPKG.oxygenMgPerL,
        EMPKG.biomTotalReads,
        EMPKG.biomObservedASVs,
        SCHEMA.latitude,
        SCHEMA.longitude,
        EMPKG.depthM,
        EMPKG.altitudeM,
        EMPKG.elevationM,
    ],
)


https://w3id.org/empkg/ontology/temperatureDegC
  rdflib.term.Literal('5.2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#decimal')) -> http://www.w3.org/2001/XMLSchema#decimal

https://w3id.org/empkg/ontology/ph
  rdflib.term.Literal('7.8', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#decimal')) -> http://www.w3.org/2001/XMLSchema#decimal

https://w3id.org/empkg/ontology/salinityPsu
  rdflib.term.Literal('30.0', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#decimal')) -> http://www.w3.org/2001/XMLSchema#decimal

https://w3id.org/empkg/ontology/oxygenMgPerL
  rdflib.term.Literal('13.2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#decimal')) -> http://www.w3.org/2001/XMLSchema#decimal

https://w3id.org/empkg/ontology/biomTotalReads
  rdflib.term.Literal('5000', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')) -> http://www.w3.org/2001/XMLSchema#integer

https://w3id.org/empkg/ontology/biomObserv

## 15. Conversión completa de las 17 muestras

Cuando la prueba de una muestra funciona, se procesa todo el DataFrame.

En el bucle principal solo se llama a `add_sample`, porque esta función ya se encarga de:

```text
crear/reutilizar Study
crear/reutilizar Location
crear/reutilizar EMPOCategory
crear EnvironmentDescription
crear Sample
crear las 4 relaciones principales
añadir literales propios de Sample
```

Aunque varias filas intenten añadir los mismos triples de `Study`, `Location` o `EMPOCategory`, RDFLib mantiene el grafo como conjunto de triples, por lo que no se generan duplicados reales.

In [77]:
g = create_empty_graph()
add_model_layer(g)

for _, row in df.iterrows():
    add_sample(g, row)

print(f"Triples generados: {len(g)}")

Triples generados: 588


## 16. Validaciones del grafo

Validaciones mínimas esperadas:

- 17 `empkg:Sample`
- 17 relaciones `empkg:belongsToStudy`
- 17 relaciones `empkg:wasCollectedAt`
- 17 relaciones `empkg:hasEMPOCategory`
- 17 relaciones `empkg:hasEnvironmentDescription`
- 17 `empkg:EnvironmentDescription`
- 17 `empkg:EMPOCategory`
- Ningún literal `"nan"`
- Ningún literal `"None"`
- Ningún literal vacío

Los números de `Study` y `Location` pueden ser menores que 17 si hay nodos reutilizados.

In [78]:
def count_subjects_of_type(graph: Graph, rdf_class: URIRef) -> int:
    return len(set(graph.subjects(RDF.type, rdf_class)))


def count_triples_with_predicate(graph: Graph, predicate: URIRef) -> int:
    return len(list(graph.triples((None, predicate, None))))

In [79]:
summary = {
    "triples_total": len(g),
    "samples": count_subjects_of_type(g, EMPKG.Sample),
    "studies": count_subjects_of_type(g, EMPKG.Study),
    "locations": count_subjects_of_type(g, EMPKG.Location),
    "empo_categories": count_subjects_of_type(g, EMPKG.EMPOCategory),
    "environment_descriptions": count_subjects_of_type(g, EMPKG.EnvironmentDescription),
    "belongs_to_study": count_triples_with_predicate(g, EMPKG.belongsToStudy),
    "was_collected_at": count_triples_with_predicate(g, EMPKG.wasCollectedAt),
    "has_empo_category": count_triples_with_predicate(g, EMPKG.hasEMPOCategory),
    "has_environment_description": count_triples_with_predicate(g, EMPKG.hasEnvironmentDescription),
    "bad_literals": len(find_bad_literals(g)),
}

summary

{'triples_total': 588,
 'samples': 17,
 'studies': 15,
 'locations': 15,
 'empo_categories': 17,
 'environment_descriptions': 17,
 'belongs_to_study': 17,
 'was_collected_at': 17,
 'has_empo_category': 17,
 'has_environment_description': 17,
 'bad_literals': 0}

In [80]:
assert summary["samples"] == 17
assert summary["studies"] == df["study_id"].nunique()
assert summary["empo_categories"] == df["empo_3"].nunique()
assert summary["environment_descriptions"] == 17

assert summary["belongs_to_study"] == 17
assert summary["was_collected_at"] == 17
assert summary["has_empo_category"] == 17
assert summary["has_environment_description"] == 17

assert summary["bad_literals"] == 0

print("OK: validaciones principales superadas.")

OK: validaciones principales superadas.


## 17. Serialización a Turtle

Cuando las validaciones pasen, serializa el grafo a:

```text
data/processed/empkg_v0_test.ttl
```

Después, vuelve a cargar el Turtle para comprobar que se puede parsear sin errores.

In [81]:
OUTPUT_TTL_PATH.parent.mkdir(parents=True, exist_ok=True)

g.serialize(destination=str(OUTPUT_TTL_PATH), format="turtle")

print(f"Turtle generado en: {OUTPUT_TTL_PATH}")
print(f"Triples serializados: {len(g)}")
print(f"Tamaño del fichero: {OUTPUT_TTL_PATH.stat().st_size:,} bytes")

Turtle generado en: /home/oier/EMPKG/data/processed/empkg_v0_test.ttl
Triples serializados: 588
Tamaño del fichero: 24,946 bytes


In [82]:
g_reloaded = create_empty_graph()
g_reloaded.parse(str(OUTPUT_TTL_PATH), format="turtle")

print(f"Triples leídos desde Turtle: {len(g_reloaded)}")

assert len(g_reloaded) == len(g)

print("OK: el Turtle generado se puede parsear correctamente con RDFLib.")

Triples leídos desde Turtle: 588
OK: el Turtle generado se puede parsear correctamente con RDFLib.


## 18. Comparación con el Turtle manual

Esta sección no pretende que el Turtle automático sea idéntico carácter por carácter al Turtle escrito a mano.

Lo importante es comprobar que el patrón RDF se mantiene:

- misma clase para la muestra;
- mismas 4 relaciones principales;
- mismos nodos conceptuales (`Study`, `Location`, `EMPOCategory`, `EnvironmentDescription`);
- mismos datatypes para literales numéricos;
- ausencia de literales `"nan"`, `"None"` o vacíos.

Es normal que alguna URI, especialmente la de `Location`, no coincida exactamente si el constructor automático usa una clave más explícita que la versión manual.

In [83]:
def find_existing_manual_ttl_path() -> Path | None:
    """
    Busca el Turtle manual usado como referencia, si está disponible.
    """
    candidate_paths = [
        PROJECT_ROOT / "data" / "samples" / "csv_to_rdf.ttl",
        PROJECT_ROOT / "data" / "processed" / "csv_to_rdf.ttl",
        PROJECT_ROOT / "csv_to_rdf.ttl",
    ]

    for path in candidate_paths:
        if path.exists():
            return path

    return None


def qname_or_str(graph: Graph, term) -> str:
    """
    Devuelve un nombre prefijado si RDFLib puede calcularlo.
    Si no, devuelve la URI/literal como texto.
    """
    try:
        return graph.qname(term)
    except Exception:
        return str(term)


def print_subject_triples(graph: Graph, subject: URIRef) -> None:
    """
    Imprime todos los triples de un sujeto de forma legible.
    """
    for predicate, obj in sorted(graph.predicate_objects(subject), key=lambda item: str(item[0])):
        print(f"  {qname_or_str(graph, predicate)} -> {qname_or_str(graph, obj)}")


manual_ttl_path = find_existing_manual_ttl_path()
manual_sample_id = "1883.2011.491.Crump.Artic.LTREB.main.lane4.NoIndex"
manual_sample_uri = make_sample_uri(manual_sample_id)

print("Muestra de referencia:", manual_sample_id)
print("URI generada:", qname_or_str(g, manual_sample_uri))

print("\nTriples automáticos de la muestra:")
print_subject_triples(g, manual_sample_uri)

if manual_ttl_path is None:
    print("\nAVISO: no se ha encontrado csv_to_rdf.ttl. Se omite la comparación con el Turtle manual.")
else:
    manual_graph = create_empty_graph()
    manual_graph.parse(str(manual_ttl_path), format="turtle")

    print(f"\nTurtle manual encontrado en: {manual_ttl_path}")
    print(f"Triples del Turtle manual: {len(manual_graph)}")

    print("\nTriples manuales de la muestra:")
    print_subject_triples(manual_graph, manual_sample_uri)

    auto_predicates = set(g.predicates(manual_sample_uri, None))
    manual_predicates = set(manual_graph.predicates(manual_sample_uri, None))

    print("\nComparación de predicados de la muestra:")
    print("Predicados comunes:", len(auto_predicates & manual_predicates))
    print("Solo en automático:", sorted(qname_or_str(g, p) for p in auto_predicates - manual_predicates))
    print("Solo en manual:", sorted(qname_or_str(manual_graph, p) for p in manual_predicates - auto_predicates))

Muestra de referencia: 1883.2011.491.Crump.Artic.LTREB.main.lane4.NoIndex
URI generada: sample:1883_2011_491_Crump_Artic_LTREB_main_lane4_NoIndex

Triples automáticos de la muestra:
  rdf:type -> empkg:Sample
  rdfs:label -> Sample 1883.2011.491.Crump.Artic.LTREB.main.lane4.NoIndex
  empkg:belongsToStudy -> study:1883
  empkg:biomObservedASVs -> 94
  empkg:biomTotalReads -> 5000
  empkg:hasEMPOCategory -> empo:water_saline
  empkg:hasEnvironmentDescription -> envdesc:1883_2011_491_Crump_Artic_LTREB_main_lane4_NoIndex
  empkg:oxygenMgPerL -> 13.2
  empkg:ph -> 7.8
  empkg:salinityPsu -> 30.0
  empkg:sampleId -> 1883.2011.491.Crump.Artic.LTREB.main.lane4.NoIndex
  empkg:temperatureDegC -> 5.2
  empkg:wasCollectedAt -> loc:united_states_of_america_lat70_06601667_lonm143_1903833_depth0_5_alt0_0_elev0_0

Turtle manual encontrado en: /home/oier/EMPKG/data/samples/csv_to_rdf.ttl
Triples del Turtle manual: 53

Triples manuales de la muestra:
  rdf:type -> empkg:Sample
  rdfs:label -> Sample 18

## 19. Conclusiones de la fase notebook

### Qué ha funcionado

- Se ha construido un primer RDF/Turtle de prueba a partir de `kg_v0_test_samples.csv`.
- El CSV de entrada tiene 17 muestras y 24 columnas esperadas.
- El subconjunto cubre las 17 categorías `empo_3`.
- El grafo generado contiene:
  - 17 nodos `empkg:Sample`;
  - 15 nodos `empkg:Study`;
  - 15 nodos `empkg:Location`;
  - 17 nodos `empkg:EMPOCategory`;
  - 17 nodos `empkg:EnvironmentDescription`.
- El número de estudios y localizaciones es menor que 17 porque hay nodos reutilizados, lo cual confirma que el modelo no crea copias innecesarias.
- Cada muestra tiene las 4 relaciones principales:
  - `empkg:belongsToStudy`;
  - `empkg:wasCollectedAt`;
  - `empkg:hasEMPOCategory`;
  - `empkg:hasEnvironmentDescription`.
- El Turtle generado se puede parsear de nuevo con RDFLib.
- No aparecen literales problemáticos como `"nan"`, `"None"`, `"null"` o cadenas vacías.

### Qué decisiones quedan fijadas

- `Sample` es el nodo central del KG v0.
- `Study`, `Location` y `EMPOCategory` son nodos reutilizables.
- `EnvironmentDescription` se mantiene 1:1 con `Sample` en v0.
- `study_id` se usa como identificador principal de `Study`; no se asume que haya un estudio por muestra.
- `empo_3` se usa como identificador principal de `EMPOCategory`.
- Los campos `env_*` y `envo_biome_*` se conservan como literales originales, sin mapear todavía a ENVO real.
- Los campos fisicoquímicos y geográficos solo generan triples si tienen valor.
- Las magnitudes numéricas se representan con tipos RDF:
  - `xsd:decimal` para coordenadas, profundidad, altitud, elevación, pH, temperatura, salinidad y oxígeno;
  - `xsd:integer` para `biom_total_reads` y `biom_observed_asvs`.

### Qué problemas han aparecido y cómo se han resuelto

- RDFLib puede serializar números con sintaxis abreviada de Turtle, por lo que no siempre se ve explícitamente `^^xsd:decimal` o `^^xsd:integer`. Se ha comprobado el datatype directamente sobre los objetos `Literal`.
- Convertir valores a `str(...)` antes de comprobar nulos puede producir literales incorrectos como `"nan"`. Se ha centralizado la lógica en helpers como `add_text_literal_if_present`, `add_decimal_literal_if_present` y `add_integer_literal_if_present`.
- Al principio se podía pensar que habría 17 estudios, pero el subconjunto tiene 15 `study_id` únicos. Esto es correcto porque varias muestras pueden pertenecer al mismo estudio.
- No es necesario comprobar manualmente si un triple ya existe antes de añadirlo, porque `rdflib.Graph` se comporta como un conjunto de triples.

### Qué se pasará después a script reproducible

La lógica estable de este notebook puede consolidarse en `scripts/03_to_rdf.py`:

- definición de namespaces;
- creación del grafo;
- capa mínima de modelo RDF;
- funciones de limpieza y sanitización;
- constructores de URIs;
- helpers para literales;
- funciones `add_study`, `add_location`, `add_empo_category`, `add_environment_description` y `add_sample`;
- construcción completa del RDF;
- validaciones del grafo;
- serialización a Turtle;
- parseo final del Turtle generado.

### Qué queda fuera de v0

Quedan fuera de esta fase:

- ASVs;
- taxones;
- abundancias muestra-ASV;
- nodos `Taxon` o `TaxonOccurrence`;
- mapeo real a ENVO;
- mapeo real a GAZ;
- mapeo real a NCBITaxon;
- validación SHACL;
- carga en GraphDB;
- consultas SPARQL avanzadas;
- procesamiento LLM.

### Siguiente paso recomendado

El siguiente paso es convertir este notebook en un script reproducible (`scripts/03_to_rdf.py`) manteniendo el notebook como documentación didáctica del proceso.